## Detect errors in executed experiments

Optionally, delete files

Designed in Norway, Written by Claude

In [8]:
import re
import pandas as pd
from pathlib import Path
from datetime import datetime

In [9]:
ROOT_DIR = "/Volumes/Disk2/data/tmp"

DELETE_ERRORS = True

In [10]:
_TXT_RE      = re.compile(r"_(\d{8})_(\d{6})\.txt$")
_PCAP_REF_RE = re.compile(r"Packets saved to '(.+?\.pcap)'")

def parse_txt_ts(path: Path):
    m = _TXT_RE.search(path.name)
    return datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S") if m else None

ERROR_PATTERNS = re.compile(
    r"Traceback \(most recent call last\)"
    r"|^\w*Error[:\s]"
    r"|^\w*Exception[:\s]"
    r"|raise \w",
    re.MULTILINE,
)

def detect_error(text: str) -> tuple[bool, str]:
    """Return (has_error, error_label)."""
    if "=== End of Run ===" not in text:
        return True, "incomplete_run"
    stderr_match = re.search(r"--- STDERR ---\n(.+)", text, re.DOTALL)
    stderr = stderr_match.group(1).strip() if stderr_match else ""
    if not stderr:
        return False, ""
    if ERROR_PATTERNS.search(stderr):
        cls = re.search(r"(\w+Error|\w+Exception)", stderr)
        return True, cls.group(1) if cls else "error"
    return True, "stderr"

def resolve_pcap(text: str, pcap_subdir: Path) -> tuple[Path | None, bool]:
    """Return (pcap_path, pcap_missing).  pcap_path is None when missing."""
    m = _PCAP_REF_RE.search(text)
    if not m:
        return None, True
    candidate = pcap_subdir / Path(m.group(1)).name
    return (candidate, False) if candidate.exists() else (None, True)

In [11]:
root = Path(ROOT_DIR)
output_root = root / "output"
pcap_root   = root / "pcap"

rows = []

for out_subdir in sorted(output_root.iterdir()):
    if not out_subdir.is_dir():
        continue
    pcap_subdir = pcap_root / out_subdir.name
    if not pcap_subdir.is_dir():
        print(f"WARNING: no matching pcap subdir for {out_subdir.name}")

    for txt_path in sorted(out_subdir.glob("*.txt"), key=parse_txt_ts):
        stem  = txt_path.stem
        parts = stem.rsplit("_", 2)
        backend_model = parts[0] if len(parts) == 3 else stem

        text = txt_path.read_text(encoding="utf-8", errors="replace")
        has_error, error_type = detect_error(text)

        pcap_path, pcap_missing = resolve_pcap(text, pcap_subdir)
        if pcap_missing and not has_error:
            has_error  = True
            error_type = "pcap_missing"

        rows.append({
            "subdir":       out_subdir.name,
            "backend":      backend_model,
            "ts_txt":       parse_txt_ts(txt_path),
            "txt_file":     txt_path.name,
            "pcap_file":    pcap_path.name if pcap_path else None,
            "pcap_missing": pcap_missing,
            "has_error":    has_error,
            "error_type":   error_type,
        })

df = pd.DataFrame(rows)
print(f"{len(df)} runs  |  {df['has_error'].sum()} errors  ({df['has_error'].mean():.1%})")
print(f"  pcap_missing: {df['pcap_missing'].sum()}")
display(df)

86 runs  |  23 errors  (26.7%)
  pcap_missing: 1


,subdir,backend,ts_txt,txt_file,pcap_file,pcap_missing,has_error,error_type
0,run_random_planner,gemini_default,2026-04-22 10:21:15,gemini_default_20260422_102115.txt,research-assistant-gemini-gemini-2.5-pro-20260...,False,False,
1,run_random_planner,deepseek_default,2026-04-22 10:22:10,deepseek_default_20260422_102210.txt,research-assistant-deepseek-deepseek-chat-2026...,False,True,incomplete_run
2,run_random_planner,openai_default,2026-04-22 11:19:03,openai_default_20260422_111903.txt,research-assistant-openai-gpt-5-20260422-11245...,False,False,
3,run_random_planner,gemini_default,2026-04-22 11:24:53,gemini_default_20260422_112453.txt,research-assistant-gemini-gemini-2.5-pro-20260...,False,False,
4,run_random_planner,deepseek_default,2026-04-22 11:26:09,deepseek_default_20260422_112609.txt,weather-deepseek-deepseek-chat-20260422-112613...,False,True,incomplete_run
...,...,...,...,...,...,...,...,...
81,run_random_planner,gemini_default,2026-04-22 14:04:10,gemini_default_20260422_140410.txt,research-assistant-gemini-gemini-2.5-pro-20260...,False,True,incomplete_run
82,run_random_planner,deepseek_default,2026-04-22 14:04:32,deepseek_default_20260422_140432.txt,no-tools-deepseek-deepseek-chat-20260422-14054...,False,False,
83,run_random_planner,openai_default,2026-04-22 14:05:44,openai_default_20260422_140544.txt,weather-openai-gpt-5-20260422-140724.pcap,False,False,
84,run_random_planner,gemini_default,2026-04-22 14:07:25,gemini_default_20260422_140725.txt,no-tools-gemini-gemini-2.5-pro-20260422-140803...,False,False,


In [12]:
print("Errors by backend:")
display(df[df["has_error"]].groupby(["backend", "error_type"]).size().rename("count").reset_index())

print("\nAll error rows:")
display(df[df["has_error"]][["subdir", "backend", "ts_txt", "txt_file", "pcap_file", "error_type"]])

Errors by backend:


,backend,error_type,count
0,deepseek_default,incomplete_run,19
1,gemini_default,incomplete_run,2
2,openai_default,incomplete_run,2



All error rows:


,subdir,backend,ts_txt,txt_file,pcap_file,error_type
1,run_random_planner,deepseek_default,2026-04-22 10:22:10,deepseek_default_20260422_102210.txt,research-assistant-deepseek-deepseek-chat-2026...,incomplete_run
4,run_random_planner,deepseek_default,2026-04-22 11:26:09,deepseek_default_20260422_112609.txt,weather-deepseek-deepseek-chat-20260422-112613...,incomplete_run
5,run_random_planner,openai_default,2026-04-22 11:26:13,openai_default_20260422_112613.txt,research-assistant-openai-gpt-5-20260422-11313...,incomplete_run
7,run_random_planner,deepseek_default,2026-04-22 11:33:37,deepseek_default_20260422_113337.txt,no-tools-deepseek-deepseek-chat-20260422-11334...,incomplete_run
10,run_random_planner,deepseek_default,2026-04-22 11:35:32,deepseek_default_20260422_113532.txt,no-tools-deepseek-deepseek-chat-20260422-11353...,incomplete_run
13,run_random_planner,deepseek_default,2026-04-22 11:39:25,deepseek_default_20260422_113925.txt,research-assistant-deepseek-deepseek-chat-2026...,incomplete_run
16,run_random_planner,deepseek_default,2026-04-22 11:49:59,deepseek_default_20260422_114959.txt,research-assistant-deepseek-deepseek-chat-2026...,incomplete_run
19,run_random_planner,deepseek_default,2026-04-22 12:02:29,deepseek_default_20260422_120229.txt,research-assistant-deepseek-deepseek-chat-2026...,incomplete_run
22,run_random_planner,deepseek_default,2026-04-22 12:04:17,deepseek_default_20260422_120417.txt,research-assistant-deepseek-deepseek-chat-2026...,incomplete_run
25,run_random_planner,deepseek_default,2026-04-22 12:06:48,deepseek_default_20260422_120648.txt,no-tools-deepseek-deepseek-chat-20260422-12065...,incomplete_run


In [13]:
if DELETE_ERRORS:
    errors = df[df["has_error"]]
    for _, row in errors.iterrows():
        txt = output_root / row["subdir"] / row["txt_file"]
        txt.unlink(missing_ok=True)
        if row["pcap_file"]:
            (pcap_root / row["subdir"] / row["pcap_file"]).unlink(missing_ok=True)
    print(f"Deleted {len(errors)} txt files (+ pcap where present).")
else:
    print("DELETE_ERRORS=False — nothing deleted.")

Deleted 23 txt files (+ pcap where present).
